# Momentum Trading Strategy Using Machine Learning

Hackathon notebook for the IIT Mandi Xpecto '26 problem statement.

**Objective:** predict whether each stock will generate a positive return over the next week, rank the universe by predicted probability, select the top 2 stocks weekly, and backtest an equal-weight long-only portfolio before and after transaction costs.

## 1. Setup

The reusable implementation lives in `src/momentum_strategy.py` so the notebook remains readable and reproducible.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.momentum_strategy import (
    RAW_TICKERS,
    FEATURE_COLUMNS,
    StrategyConfig,
    download_daily_data,
    build_weekly_dataset,
    train_predict,
    construct_portfolio,
    backtest_portfolio,
)

sns.set_theme(style='whitegrid')
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

config = StrategyConfig()
RAW_TICKERS

## 2. Data Download and Cleaning

Daily OHLCV data is downloaded from Yahoo Finance for 2017 through 2025. Prices are auto-adjusted by `yfinance`, so the `close` column is adjusted for splits and dividends.

In [ ]:
daily = download_daily_data(RAW_TICKERS, config.start, config.end)
daily.to_csv(OUTPUT_DIR / 'daily_ohlcv.csv', index=False)

print(daily.shape)
daily.head()

In [ ]:
daily.groupby('ticker').agg(
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    rows=('date', 'size'),
    missing_close=('close', lambda s: s.isna().sum()),
).sort_index()

## 3. Feature Engineering

Features are computed weekly using only information available at the rebalance date. The target is `1` when the next-week close-to-close return is positive, otherwise `0`.

In [ ]:
weekly_dataset = build_weekly_dataset(daily)
weekly_dataset.to_csv(OUTPUT_DIR / 'weekly_features.csv', index=False)

print(weekly_dataset.shape)
weekly_dataset[['week', 'ticker', *FEATURE_COLUMNS, 'next_week_return', 'target']].head()

In [ ]:
weekly_dataset.groupby('target').size().rename('count').to_frame()

## 4. Model Training and Prediction

The default model is a Random Forest classifier. It is trained on 2017-2022 and evaluated out of sample on 2023-2025.

In [ ]:
model, predictions, model_metrics = train_predict(weekly_dataset, config)
model_metrics

In [ ]:
portfolio_rows = construct_portfolio(predictions, config)
portfolio_rows.to_csv(OUTPUT_DIR / 'weekly_stock_predictions.csv', index=False)

portfolio_rows[['week', 'ticker', 'predicted_probability', 'rank', 'selected', 'weight', 'next_week_return']].head(20)

## 5. Portfolio Construction and Backtest

Each week, the top 2 stocks by predicted probability are selected with 50% weight each. Gross return is measured before costs. Net return subtracts 0.1% entry cost and 0.1% exit cost per weekly rebalance.

In [ ]:
weekly_returns, performance = backtest_portfolio(portfolio_rows, config)
weekly_returns.to_csv(OUTPUT_DIR / 'weekly_portfolio_returns.csv', index=False)
performance.to_csv(OUTPUT_DIR / 'performance_metrics.csv', index=False)

performance

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(weekly_returns['week'], weekly_returns['gross_equity'], label='Before costs')
ax.plot(weekly_returns['week'], weekly_returns['net_equity'], label='After costs')
ax.set_title('ML Momentum Strategy Equity Curve')
ax.set_ylabel('Growth of $1')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'equity_curve.png', dpi=160)
plt.show()

In [ ]:
gross_drawdown = weekly_returns['gross_equity'] / weekly_returns['gross_equity'].cummax() - 1
net_drawdown = weekly_returns['net_equity'] / weekly_returns['net_equity'].cummax() - 1

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(weekly_returns['week'], gross_drawdown, 0, alpha=0.35, label='Before costs')
ax.fill_between(weekly_returns['week'], net_drawdown, 0, alpha=0.35, label='After costs')
ax.set_title('Drawdown')
ax.set_ylabel('Drawdown')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'drawdown.png', dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(weekly_returns['net_return'], bins=30, kde=True, ax=ax)
ax.set_title('After-Cost Weekly Return Distribution')
ax.set_xlabel('Weekly return')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'weekly_return_distribution.png', dpi=160)
plt.show()

In [ ]:
selection_counts = portfolio_rows.loc[portfolio_rows['selected'], 'ticker'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
selection_counts.plot(kind='barh', ax=ax)
ax.set_title('Selection Frequency by Stock')
ax.set_xlabel('Selected weeks')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'selection_frequency.png', dpi=160)
plt.show()

In [ ]:
rf = model.named_steps['model']
feature_importance = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
feature_importance.plot(kind='barh', ax=ax)
ax.set_title('Random Forest Feature Importance')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=160)
plt.show()

feature_importance.sort_values(ascending=False).to_frame('importance')

## 7. Deliverable CSV Preview

In [ ]:
deliverable_columns = [
    'week', 'ticker', 'predicted_probability', 'rank', 'selected', 'weight',
    'next_week_return', 'return_contribution'
]
portfolio_rows[deliverable_columns].head(30)

## 8. Summary

Use `outputs/performance_metrics.csv` for the final report values. The core comparison is before-cost versus after-cost performance, with special attention to Sharpe ratio, max drawdown, hit rate, and whether transaction costs materially reduce the apparent edge.